# NeuroTrain Lab — Notebook 1: The Perceptron

**Topic:** what an artificial neuron is, what activation functions (ReLU,
Sigmoid, Softmax) are for, how they stack into a multi-layer network (MLP), and
how all of this looks as tensors in PyTorch and TensorFlow.

> First of 4 notebooks. We don't train anything yet — we're only understanding
> how a network **predicts**. Teaching it to make fewer mistakes is Notebook 2.

## 🎯 What you'll learn in this notebook

By the end you should be able to explain, without memorized formulas:

1. What a neuron (perceptron) computes mathematically.
2. What ReLU, Sigmoid, and Softmax do, and when to use each.
3. Why a single neuron isn't enough, and why we stack layers (MLP).
4. Why "forward propagation" is, underneath, just matrix multiplication.
5. What a tensor is, and how the same computation looks in NumPy, PyTorch, and TensorFlow.

**Mental map:** `neuron → activation → layer → MLP → forward propagation → tensors`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from sklearn.datasets import make_moons

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.visualization import plot_decision_boundary

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| PyTorch:", torch.__version__, "| TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## 0. The thread running through all 4 notebooks

All 4 notebooks share the same real underlying problem: **predicting whether a
tumor is benign or malignant** from 30 numeric features (the *Breast Cancer
Wisconsin* dataset). Here we just glance at it — the full audit and real training
happen in Notebook 4.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
df.head(3)

## 1. What an artificial neuron is

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — A neuron as a nightclub bouncer</b><br><br>
Picture a bouncer deciding whether to let someone in. They don't look at just
one thing: they weigh several signals (well dressed? on the guest list? how late
is it?), and each signal matters a different amount to them. If the weighted sum
clears their threshold for the night (how strict they're feeling), you're in.

An artificial neuron does exactly that with numbers: it multiplies each input
`x` by a **weight** saying how much it matters, adds a **bias** (how strict the
neuron is by default), and applies an activation function to decide the output.
</div>

Mathematically, for a neuron with inputs $x_1, x_2, \dots, x_n$:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

$z$ is just a number. Turning it into a decision needs an **activation
function** — that's Section 2. First, let's compute $z$ by hand.

In [ ]:
def weighted_sum(x, w, b):
    """z = x·w + b — what a neuron computes before activating."""
    return np.dot(x, w) + b


# Example: "should I bring an umbrella?" — 2 inputs: rain probability, wind (0-1)
x_example = np.array([0.8, 0.3])
w_example = np.array([0.9, 0.2])
b_example = -0.4

z = weighted_sum(x_example, w_example, b_example)
print("z (weighted sum):", z)

### ✏️ Exercise

Complete `weighted_sum_manual`, computing `z` **without using `np.dot`**, using a
`for` loop over `x` and `w` together. It should match the previous cell
(`z ≈ 0.98`).

In [ ]:
def weighted_sum_manual(x, w, b):
    total = 0.0
    for xi, wi in zip(x, w):
        total += ✏️✏️✏️
    return total + b

print(weighted_sum_manual(x_example, w_example, b_example))

<details>
<summary><b>Show solution</b></summary>

```python
def weighted_sum_manual(x, w, b):
    total = 0.0
    for xi, wi in zip(x, w):
        total += xi * wi
    return total + b

print(weighted_sum_manual(x_example, w_example, b_example))
```

</details>

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — Which weight does the neuron keep?</b><br><br>
All of them. If a neuron receives 30 values (like in our real dataset), it has
**30 weights** — one per input — plus a bias. There's no such thing as "the
neuron's weight" in the singular; every incoming connection has its own.
</div>

## 2. Activation functions: ReLU, Sigmoid, and Softmax

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — Why not just leave z as it is?</b><br><br>
If we stack neurons with no non-linear activation in between, the whole
network — no matter how many layers — stays mathematically equivalent to a
**single** linear transformation. Non-linearity is what lets a network learn
curved shapes, not just straight lines. We'll prove it in Section 2.3.
</div>

In [ ]:
def relu(z):
    return np.maximum(0, z)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


z_values = np.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(z_values, relu(z_values), color="#7C3AED")
axes[0].set_title("ReLU(z) = max(0, z)")
axes[0].axhline(0, color="#94A3B8", linewidth=0.8)
axes[1].plot(z_values, sigmoid(z_values), color="#2563EB")
axes[1].set_title("Sigmoid(z) = 1 / (1 + e⁻ᶻ)")
axes[1].axhline(0.5, color="#94A3B8", linewidth=0.8, linestyle="--")
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print("ReLU(-3) =", relu(-3), "| ReLU(2) =", relu(2))
print("Sigmoid(0) =", sigmoid(0), "| Sigmoid(6) ≈", round(sigmoid(6), 3))

- **ReLU** says "pass through unchanged if you're positive, zero out if you're
  negative." Used almost everywhere in **hidden layers**: cheap to compute and
  helps gradients flow well (more on that in Notebook 3).
- **Sigmoid** squashes any number into the range (0, 1) — read as a
  **probability**. Used in the **output layer** for binary classification
  (malignant or benign? a single number between 0 and 1).

### ✏️ Exercise

Implement `sigmoid` and `relu` yourself and check they match Keras.

In [ ]:
def my_relu(z):
    return np.maximum(✏️✏️✏️, z)


def my_sigmoid(z):
    return 1 / (1 + np.exp(✏️✏️✏️))


keras_relu = tf.keras.activations.relu(tf.constant([-2.0, 0.0, 3.0])).numpy()
keras_sigmoid = tf.keras.activations.sigmoid(tf.constant([-2.0, 0.0, 3.0])).numpy()

assert np.allclose(my_relu(np.array([-2.0, 0.0, 3.0])), keras_relu)
assert np.allclose(my_sigmoid(np.array([-2.0, 0.0, 3.0])), keras_sigmoid)
print("Matches Keras!")

<details>
<summary><b>Show solution</b></summary>

```python
def my_relu(z):
    return np.maximum(0, z)


def my_sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

</details>

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ TYPICAL MISTAKE — Non-linearity actually matters</b><br><br>
Let's check what happens if you chain two **linear** transformations with no
activation in between: it's still a single linear transformation, no new curves.
</div>

In [ ]:
# Two "linear" layers (no activation) composed are still just one line
A = np.array([[2.0, 0.0], [0.0, 2.0]])
B = np.array([[1.0, 1.0], [0.0, 1.0]])
composed = A @ B
print("Applying A then B is equivalent to ONE single matrix:")
print(composed)
print("That's why we insert ReLU/Sigmoid between layers: they break that equivalence.")

### Softmax: when there are more than 2 classes

Sigmoid gives one probability for one class. When there are **several mutually
exclusive classes** (e.g. "apple / banana / orange"), we use **Softmax**: it
turns a list of numbers (*logits*) into probabilities that **always sum to 1**.

In [ ]:
def softmax(logits):
    exponents = np.exp(logits - np.max(logits))  # -max: numerical stability
    return exponents / exponents.sum()


fruit_logits = np.array([2.0, 1.0, 0.1])  # raw scores for apple/banana/orange
probabilities = softmax(fruit_logits)
print("Logits:       ", fruit_logits)
print("Probabilities:", probabilities.round(3))
print("Sum:          ", probabilities.sum())

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🌱 You just planted the first seed of your neural network.</div>

## 3. From one neuron to an MLP

A single neuron can only draw a **straight line** boundary. To learn more
complex shapes, we stack neurons into **layers**, and layers into a
**Multi-Layer Perceptron (MLP)**:

`inputs → hidden layer 1 → hidden layer 2 → ... → output layer`

The number of neurons per layer (32, 16, whatever) and the number of layers are
**hyperparameters**: decisions we make and validate, not formulas derived from
the number of inputs.

## 4. Forward propagation as matrix multiplication

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — A Dense layer is just this</b><br><br>
`Dense(n, activation)` computes, for **every example in the batch at once**:

$$H = \text{activation}(X \cdot W + b)$$

$X$ is the input matrix, $W$ is the layer's weight matrix (one column per
neuron), and $b$ is the bias vector. It's the exact same computation we did by
hand in Section 1, applied to every neuron in the layer at once.
</div>

In [ ]:
# A toy MLP: 2 inputs -> hidden layer of 3 -> output of 1
x = np.array([[1.0, 2.0]])  # 1 example, 2 features

W1 = np.array([[0.5, -0.9, 0.3],
               [0.2,  0.1, -0.1]])   # (2 inputs, 3 hidden neurons)
b1 = np.array([0.1, -0.2, 0.05])

z1 = x @ W1 + b1
h1 = relu(z1)
print("z1 (before activation):", z1)
print("h1 (after ReLU):       ", h1, "  <- the -0.9 became 0")

W2 = np.array([[0.7], [-0.5], [0.9]])  # (3 inputs, 1 output neuron)
b2 = np.array([-0.1])

z2 = h1 @ W2 + b2
y_hat = sigmoid(z2)
print("z2 (before activation):", z2)
print("y_hat (after Sigmoid): ", y_hat, " <- final probability")

### ✏️ Exercise

Repeat the forward pass above for a second example `x2 = [[-1.0, 0.5]]`, reusing
the same `W1`, `b1`, `W2`, `b2`. Fill in the first layer's matrix multiplication.

In [ ]:
x2 = np.array([[-1.0, 0.5]])

z1_b = ✏️✏️✏️ + b1
h1_b = relu(z1_b)
z2_b = h1_b @ W2 + b2
y_hat_b = sigmoid(z2_b)
print("y_hat for x2:", y_hat_b)

<details>
<summary><b>Show solution</b></summary>

```python
x2 = np.array([[-1.0, 0.5]])

z1_b = x2 @ W1 + b1
h1_b = relu(z1_b)
z2_b = h1_b @ W2 + b2
y_hat_b = sigmoid(z2_b)
print("y_hat for x2:", y_hat_b)
```

</details>

## 5. Seeing why layers matter: decision boundaries

With 30 real features we can't "draw" the boundary that separates malignant from
benign. So we use a synthetic 2-feature dataset — `make_moons` — where we can.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=RANDOM_STATE)

plt.figure(figsize=(4.5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu_r", edgecolor="white")
plt.title("make_moons: 2 classes, curved boundary")
plt.show()

In [ ]:
# A single perceptron (equivalent to Dense(1, sigmoid) with no hidden layers)
perceptron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
perceptron.compile(optimizer="adam", loss="binary_crossentropy")
perceptron.fit(X_moons, y_moons, epochs=80, verbose=0)

# A small MLP with one non-linear hidden layer
mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy")
mlp.fit(X_moons, y_moons, epochs=80, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_decision_boundary(
    X_moons, y_moons,
    lambda grid: perceptron.predict(grid, verbose=0).ravel(),
    title="A perceptron: can only draw a line", lang="en", ax=axes[0],
)
plot_decision_boundary(
    X_moons, y_moons,
    lambda grid: mlp.predict(grid, verbose=0).ravel(),
    title="MLP (Dense 8, ReLU): curves the boundary", lang="en", ax=axes[1],
)
fig.tight_layout()
plt.show()

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — What about the real 30-feature dataset?</b><br><br>
Same principle, just living in a 30-dimensional space we can't draw on paper.
`make_moons` exists purely so you can **see with your own eyes** why an MLP with
a non-linear activation separates shapes a lone perceptron can't.

**Challenge:** rerun this cell swapping `make_moons` for
`sklearn.datasets.make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=RANDOM_STATE)`.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 The network is taking shape under your hands.</div>

## 6. Tensors: NumPy, PyTorch, and TensorFlow

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — A tensor is just an array with superpowers</b><br><br>
A **tensor** is the same idea as a NumPy array (numbers organized in
rows/columns/dimensions), plus two extras that will matter later: it can live on
a GPU, and it can **remember the operations applied to it** to compute gradients
automatically (Notebook 2). For today's basic operations, they behave exactly
like an array.
</div>

In [ ]:
# Create the same 2x2 tensor in all three "languages"
data = [[1.0, 2.0], [3.0, 4.0]]

array_np = np.array(data)
tensor_torch = torch.tensor(data)
tensor_tf = tf.constant(data)

print("NumPy     ->", array_np.shape, array_np.dtype)
print("PyTorch   ->", tensor_torch.shape, tensor_torch.dtype)
print("TensorFlow->", tensor_tf.shape, tensor_tf.dtype)

In [ ]:
# Same basic operations across all three frameworks
print("Add +10:")
print(" numpy :", array_np + 10)
print(" torch :", (tensor_torch + 10).numpy())
print(" tf    :", (tensor_tf + 10).numpy())

print("\nReshape to (4,):")
print(" numpy :", array_np.reshape(4))
print(" torch :", tensor_torch.reshape(4).numpy())
print(" tf    :", tf.reshape(tensor_tf, (4,)).numpy())

Now let's repeat **exactly** the forward pass from Section 4, computed in all three frameworks at once — they should give the same number.

In [ ]:
x_np = np.array([[1.0, 2.0]], dtype="float32")

# --- NumPy (what we already did) ---
numpy_output = sigmoid(relu(x_np @ W1 + b1) @ W2 + b2)

# --- PyTorch ---
x_t = torch.tensor(x_np)
W1_t, b1_t = torch.tensor(W1, dtype=torch.float32), torch.tensor(b1, dtype=torch.float32)
W2_t, b2_t = torch.tensor(W2, dtype=torch.float32), torch.tensor(b2, dtype=torch.float32)
torch_output = torch.sigmoid(torch.relu(x_t @ W1_t + b1_t) @ W2_t + b2_t)

# --- TensorFlow ---
x_tf = tf.constant(x_np)
tf_output = tf.sigmoid(tf.nn.relu(x_tf @ W1 + b1) @ W2 + b2)

print("NumPy      ->", numpy_output)
print("PyTorch    ->", torch_output.numpy())
print("TensorFlow ->", tf_output.numpy())
print("\nDo all three match?", np.allclose(numpy_output, torch_output.numpy())
      and np.allclose(numpy_output, tf_output.numpy()))

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 REMEMBER THIS — Don't memorize PyTorch syntax</b><br><br>
The rest of the project (Notebooks 2-4, the app) uses **TensorFlow/Keras** for
actual training. PyTorch shows up here — and once more in Notebook 2 — only so
you recognize the same concepts behind different syntax. You don't need to
master PyTorch to follow the rest of the course.
</div>

### ✏️ Exercise

Create a PyTorch tensor with values `[10, 20, 30, 40, 50, 60]` and reshape it to `(2, 3)`.

In [ ]:
values = torch.tensor([10, 20, 30, 40, 50, 60])
matrix = values.reshape(✏️✏️✏️)
print(matrix)
print(matrix.shape)

<details>
<summary><b>Show solution</b></summary>

```python
values = torch.tensor([10, 20, 30, 40, 50, 60])
matrix = values.reshape(2, 3)
print(matrix)
print(matrix.shape)
```

</details>

## 🎯 Self-assessment

Answer without looking back. You don't need perfect phrasing: explain the mechanism in your own words.

**1. A neuron receives 64 input values. How many weights does it have?**

A. 1, shared across all inputs
B. 64, one per input, plus the bias
C. 64, plus one more per neuron in the network
D. 0, weights belong to the layer, not the neuron

<details>
<summary><b>Show answer</b></summary>

**B.** Every incoming connection has its own weight. 64 inputs → 64 weights + 1 bias.

</details>

**2. What does ReLU output for a negative input, e.g. -3?**

A. -3 unchanged
B. 0
C. 3 (the absolute value)
D. An error, ReLU doesn't accept negatives

<details>
<summary><b>Show answer</b></summary>

**B.** ReLU(z) = max(0, z). Any negative value becomes 0; positive values pass through unchanged.

</details>

**3. Why does an MLP separate `make_moons` when a single perceptron can't?**

A. Because the MLP sees more training data
B. Because the MLP combines several neurons with a non-linear activation, enabling curved boundaries
C. Because the perceptron uses Sigmoid and the MLP doesn't
D. There's no real difference, it's down to the random seed

<details>
<summary><b>Show answer</b></summary>

**B.** A perceptron can only draw a straight line. Stacking neurons with a non-linearity in between lets that boundary curve.

</details>

**4. Softmax turns 3 logits into 3 probabilities. What's always true about the result?**

A. They're all exactly 0.33
B. They sum to exactly 1
C. The largest is always above 0.9
D. They can be negative if the logit is negative

<details>
<summary><b>Show answer</b></summary>

**B.** Softmax normalizes so every class's probability sums to 1, regardless of the input values.

</details>

**5. For a basic operation like adding 10, what's different between a PyTorch tensor and a NumPy array?**

A. The numeric result is different
B. Nothing in the result; the tensor can additionally run on GPU and track gradients
C. Tensors don't support reshape
D. Tensors only accept integers

<details>
<summary><b>Show answer</b></summary>

**B.** For basic operations the numeric result is identical. Tensors add capabilities (GPU, autograd) we'll use starting in Notebook 2.

</details>

**6. Explain in your own words what a forward pass is doing, without using the word 'magic'.**

<details>
<summary><b>What a good answer should include</b></summary>

- Mentions that each layer does a matrix multiplication plus a bias vector.
- Explains that after each layer (almost always except the last) there's a non-linear activation.
- Makes clear the final result is a prediction, not yet any learning.

</details>

**7. A classmate says: 'with more neurons in the hidden layer, the network always predicts better.' Do you agree?**

<details>
<summary><b>What a good answer should include</b></summary>

- Distinguishes between capacity (more parameters) and generalization (predicting well on new data).
- Mentions that more neurons also means more risk of overfitting (picked back up in Notebook 4).
- Concludes that the number of neurons is a hyperparameter to validate, not a fixed rule.

</details>

In [ ]:
celebrate(
    "🎉 Congratulations! You finished Notebook 1: The Perceptron 🎉",
    "You now know how a neuron turns numbers into decisions, and how that looks "
    "as tensors. In Notebook 2 you'll discover how the network learns from its mistakes.",
)